# 🛸 DroneNet-FPN-Attention: Google Colab Training & Interactive Demo
### ISLab Pusan National University — AI Engineer / Researcher Assignment
**Author:** Ghiffari Ahmadijaya (`ghiffariahmadijaya@gmail.com`)

**Features:**
- 🚀 **1-Click Google Drive Sync** (Never lose checkpoint weights on session disconnect)
- 📥 **Auto-Download Google Drive Dataset** (`19L9yUP62xMESJMw6srf5HGcL8s5b0gv8`)
- 🎯 **100% From-Scratch Vanilla Architecture** (Zero Pretrained Weights)
- ⚡ **Automatic Mixed Precision (AMP)** for Fast Training on Free T4/A100 GPUs
- 📈 **Live TensorBoard & Weights & Biases (W&B) Tracking**
- 🖼️ **Interactive Real-Time Drone Detection Visualizer**
- 📄 **IEEE Conference Paper Auto-Compilation**

In [ ]:
# 1. Mount Google Drive (Optional for Checkpoint Backup) & Check GPU
import os, sys, torch
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/islab_drone_project"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Google Drive mounted at: {DRIVE_DIR}")
except Exception as e:
    print("Google Drive not mounted (running in local/standalone mode)")
    DRIVE_DIR = None

!nvidia-smi
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
num_gpus = torch.cuda.device_count()
print(f"Device Count    : {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 2. Install Dependencies Quietly
!pip install -q --no-cache-dir gdown tabulate pyyaml wandb tensorboard reportlab pypdf onnx onnxruntime onnxscript

In [ ]:
# 3. Clone Repository from GitHub into /content
import os, shutil
os.chdir("/content")
if not os.path.exists("src"):
    print("Cloning codebase from GitHub: https://github.com/itanium-g/islab-pusan-ai-assignment.git...")
    !git clone https://github.com/itanium-g/islab-pusan-ai-assignment.git /tmp/repo
    !cp -r /tmp/repo/* .
    !cp -r /tmp/repo/.* . 2>/dev/null || true
    !rm -rf /tmp/repo
print("Codebase initialized successfully into /content.")
print("Directory contents:", os.listdir("/content"))

In [ ]:
# 4. Download and Extract Dataset from Google Drive
import os, zipfile, shutil, glob, gdown
GDRIVE_ID = "19L9yUP62xMESJMw6srf5HGcL8s5b0gv8"
TMP_DATA_DIR = "/tmp/curated_datasets/obj_det_base"
os.makedirs(TMP_DATA_DIR, exist_ok=True)

existing_txts = glob.glob(f"{TMP_DATA_DIR}/*.txt")
if len(existing_txts) < 100:
    print("Downloading dataset from Google Drive...")
    gdown.download(id=GDRIVE_ID, output="/tmp/dataset.zip", quiet=False)
    print("Extracting dataset.zip...")
    with zipfile.ZipFile("/tmp/dataset.zip", "r") as zf:
        zf.extractall("/tmp/extracted")
    if os.path.exists("/tmp/dataset.zip"):
        os.remove("/tmp/dataset.zip")
    # Move extracted files to TMP_DATA_DIR
    for root, dirs, files in os.walk("/tmp/extracted"):
        if any(f.endswith(".txt") for f in files):
            for f in files:
                src_p = os.path.join(root, f)
                dst_p = os.path.join(TMP_DATA_DIR, f)
                if not os.path.exists(dst_p):
                    shutil.move(src_p, dst_p)
            break
    if os.path.exists("/tmp/extracted"):
        shutil.rmtree("/tmp/extracted")

total_files = len(glob.glob(f"{TMP_DATA_DIR}/*.txt"))
print(f"Verified dataset: {total_files} annotation files in {TMP_DATA_DIR}")

# Create symlink for seamless path access
os.makedirs("curated_datasets", exist_ok=True)
if not os.path.exists("curated_datasets/obj_det_base"):
    os.symlink(TMP_DATA_DIR, "curated_datasets/obj_det_base")
print("Symlinked curated_datasets/obj_det_base ->", TMP_DATA_DIR)

In [ ]:
# 5. Preprocess & Cache Dataset (Reduces Epoch Time from 3 min to 3s!)
!python scripts/split_dataset.py --dataset-dir curated_datasets/obj_det_base --output-dir data/splits
!python scripts/preprocess_dataset.py --src-dir curated_datasets/obj_det_base --dest-dir data/cached_640 --img-size 640 --workers 4

In [ ]:
# 6. Launch Live TensorBoard Tracking
%load_ext tensorboard
%tensorboard --logdir runs/train

In [ ]:
# 7. Train Proposed Best Model (Model 3: DroneNet-FPN-Attention)
import torch
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    print(f"Launching Multi-GPU DDP Training across {num_gpus} GPUs...")
    !torchrun --nproc_per_node={num_gpus} train.py --config configs/model3_fpn_attn.yaml --epochs 40 --batch-size 16 --ddp
elif num_gpus == 1:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Launching Training on GPU ({gpu_name}) with AMP...")
    !python train.py --config configs/model3_fpn_attn.yaml --epochs 40 --batch-size 16
else:
    print("Training on CPU...")
    !python train.py --config configs/model3_fpn_attn.yaml --epochs 5 --batch-size 8

In [ ]:
# 8. Train Baseline Models (Model 1 & Model 2) for Benchmark Comparison
!python train.py --config configs/model1_baseline.yaml --epochs 30 --batch-size 16
!python train.py --config configs/model2_fpn.yaml --epochs 35 --batch-size 16

In [ ]:
# 9. Evaluate All Models on Independent Test Set
print("=== EVALUATING MODEL 1 (BASELINE) ===")
!python evaluate.py --config configs/model1_baseline.yaml --weights runs/train/model1_vanilla_baseline/checkpoints/best_model.pth --split test

print("\n=== EVALUATING MODEL 2 (MULTI-SCALE FPN) ===")
!python evaluate.py --config configs/model2_fpn.yaml --weights runs/train/model2_fpn_multiscale/checkpoints/best_model.pth --split test

print("\n=== EVALUATING MODEL 3 (FPN + ATTENTION - BEST MODEL) ===")
!python evaluate.py --config configs/model3_fpn_attn.yaml --weights runs/train/model3_fpn_attention_best/checkpoints/best_model.pth --split test

In [ ]:
# 10. Export Lightweight Weights (< 15 MB), ONNX, and TorchScript
os.makedirs("weights", exist_ok=True)
!python scripts/export_weights.py --config configs/model3_fpn_attn.yaml --checkpoint runs/train/model3_fpn_attention_best/checkpoints/best_model.pth --output-dir weights

# Backup to Google Drive if mounted
if DRIVE_DIR and os.path.exists(DRIVE_DIR):
    !cp -r weights/* {DRIVE_DIR}/
    print(f"Weights backed up to: {DRIVE_DIR}")

In [ ]:
# 11. Generate Publication Figures & Compile IEEE Conference Paper PDF
!python scripts/generate_paper_figures.py
!python scripts/compile_paper.py
if DRIVE_DIR and os.path.exists(DRIVE_DIR):
    !cp paper/Drone_Detection_Paper.pdf {DRIVE_DIR}/
    print(f"Paper PDF backed up to: {DRIVE_DIR}/Drone_Detection_Paper.pdf")

In [ ]:
# 12. Interactive Visual Inference & Sample Detections
import glob, os
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs("runs/infer", exist_ok=True)
!python infer.py --config configs/model3_fpn_attn.yaml --weights runs/train/model3_fpn_attention_best/checkpoints/best_model.pth --source data/cached_640/images --output-dir runs/infer --conf-thresh 0.35

sample_renders = glob.glob("runs/infer/*.jpg")[:4]
for p in sample_renders:
    plt.figure(figsize=(10, 6))
    plt.imshow(Image.open(p))
    plt.title(f"Colab Detection Output: {os.path.basename(p)}")
    plt.axis("off")
    plt.show()